# 🔬 22 · Image-Based Profiling and Perturbation Biology
### Turning a well of cells into a number vector — and keeping the plate from lying to you

**Part J · Research frontier** · Module 22 of 29 (frontier) · builds on 07, 08, 09, 18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/22_image_based_profiling_and_perturbation_biology.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (Colab free tier) |
| **Core frameworks** | NumPy, SciPy, Matplotlib |
| **Prerequisites** | Modules 07, 08, 09, 18 |
| **Estimated time** | 70–90 min |
| **Builds on** | Module 07 (classical feature-based classification), Module 08 (deep learning), Module 09 (dimensionality reduction) and Module 18 (spatial/quantitative analysis) |
| **Research frontier covered** | Image-based phenotypic profiling — high-content screening, per-cell feature embeddings, batch-effect correction, and hit-calling for perturbation biology |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every cell population, treatment effect and batch effect in this notebook is SIMULATED with known, controllable ground truth — there is no real screening data, and nothing is downloaded.

## 🎯 Learning objectives

By the end of this module you will be able to:

1. Explain what a morphological/phenotypic **profile** is (a feature vector per cell or per well) and why
   high-content screening treats this as the natural output, rather than a single class label.
2. Contrast classical, handcrafted per-cell features (Module 07) with learned deep-embedding features
   (Modules 08–09) as two ways of producing a profile, and state a trade-off between them.
3. Reproduce, with a controlled synthetic experiment, how a **batch effect** — a systematic, non-biological
   shift between plates or imaging sessions — can confound or even reverse the appearance of a true
   treatment effect in a profile's low-dimensional embedding.
4. Apply a simple batch-correction procedure and show, quantitatively, that it recovers the true treatment
   structure that the batch effect had obscured.
5. Describe, at a conceptual level, how **hit-calling** turns a batch-corrected profile into a decision about
   which treatments produced a significant phenotypic effect.

## 🗺️ Contents

1. The biological question
2. Environment setup
3. From an image to a profile
4. Simulating treatments, batches, and a profiling pipeline
5. Batch effects confound treatment structure
6. Correcting for batch effects
7. Hit-calling, conceptually
8. From toy to real data
9. Key takeaways, checkpoint, and further practice

## 🧫 The biological question

Drug discovery and functional genomics both ask a version of the same question at enormous scale: *"what
does this chemical compound (or this gene knockdown) do to a cell?"* Imaging cells after treatment and
extracting a rich set of morphological features — not just "alive or dead," but the full shape, texture and
organelle organisation of the cell — turns each well of a screening plate into a **profile**: a vector of
numbers that captures the cell population's phenotype. Comparing profiles across thousands of treatments
lets you cluster compounds by mechanism, flag which treatments produced a real phenotypic change, and even
predict a treatment's likely mechanism from the profile of treatments with known mechanisms that it most
resembles.

The practical obstacle that this module focuses on is not exotic biology — it is measurement discipline.
Screens run across many plates, days, and imaging sessions, and anything that differs systematically between
them (illumination drift, reagent batch, incubation time) leaves its own fingerprint in the profiles,
entirely separate from the biology. If you do not explicitly account for it, a **batch effect** can look
exactly like — or even cancel out — a genuine treatment effect.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 22
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def shape_features(mask: np.ndarray) -> dict:
    """Simple morphological/shape features of a binary blob mask: area, perimeter (boundary
    pixel count), equivalent-circle compactness, and the aspect ratio of its bounding box."""
    area = mask.sum()
    eroded = ndi.binary_erosion(mask)
    perimeter = max(1, (mask & ~eroded).sum())
    compactness = 4 * np.pi * area / (perimeter ** 2 + 1e-9)  # 1.0 for a perfect circle
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return dict(area=0, perimeter=0, compactness=0.0, aspect_ratio=1.0)
    h = ys.max() - ys.min() + 1
    w = xs.max() - xs.min() + 1
    aspect_ratio = max(h, w) / max(1, min(h, w))
    return dict(area=float(area), perimeter=float(perimeter),
                compactness=float(compactness), aspect_ratio=float(aspect_ratio))


def glcm_features(image: np.ndarray, levels: int = 8, dx: int = 1, dy: int = 0):
    """Haralick-style texture features from a grey-level co-occurrence matrix (GLCM).

    Quantises ``image`` to ``levels`` grey levels, builds the co-occurrence matrix for the
    offset (dy, dx), and returns a dict of contrast, energy (angular second moment),
    homogeneity and correlation.
    """
    img = image - image.min()
    img = img / (img.max() + 1e-9)
    q = np.clip((img * (levels - 1)).round().astype(int), 0, levels - 1)
    h, w = q.shape
    glcm = np.zeros((levels, levels))
    y0, y1 = max(0, -dy), h - max(0, dy)
    x0, x1 = max(0, -dx), w - max(0, dx)
    a = q[y0:y1, x0:x1]
    b = q[y0 + dy:y1 + dy, x0 + dx:x1 + dx]
    for i, j in zip(a.ravel(), b.ravel()):
        glcm[i, j] += 1
    glcm /= glcm.sum() + 1e-9
    i_idx, j_idx = np.meshgrid(np.arange(levels), np.arange(levels), indexing="ij")
    mu_i, mu_j = (glcm * i_idx).sum(), (glcm * j_idx).sum()
    sig_i = np.sqrt((glcm * (i_idx - mu_i) ** 2).sum()) + 1e-9
    sig_j = np.sqrt((glcm * (j_idx - mu_j) ** 2).sum()) + 1e-9
    contrast = (glcm * (i_idx - j_idx) ** 2).sum()
    energy = (glcm ** 2).sum()
    homogeneity = (glcm / (1 + np.abs(i_idx - j_idx))).sum()
    correlation = ((glcm * (i_idx - mu_i) * (j_idx - mu_j)).sum()) / (sig_i * sig_j)
    return dict(contrast=contrast, energy=energy, homogeneity=homogeneity, correlation=correlation)


def pca(X: np.ndarray, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data. Returns the projected
    scores (N x n_components), the components (n_components x J) and the explained-variance
    ratio of each component."""
    mean = X.mean(axis=0)
    Xc = X - mean
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = (U * S)[:, :n_components]
    explained = (S ** 2) / np.sum(S ** 2)
    return scores, Vt[:n_components], explained[:n_components]


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · From an image to a profile

A **profile** is a feature vector that summarises a cell, or a whole well of
cells, in a way that supports comparison across treatments. Two broad ways to build one:

- **Classical, handcrafted features** (Module 07): shape (area, compactness, aspect ratio), texture (GLCM/
  Haralick statistics), and intensity-distribution descriptors, computed per cell and then aggregated (e.g.
  averaged) across all the cells in a well. Interpretable — you can say exactly what "feature 3" measures —
  but limited to whatever the person who designed the feature set thought to include.
- **Learned deep embeddings** (Modules 08–09): the activations of an intermediate layer of a network trained
  (or pretrained) on cell images, used as a feature vector directly. Can capture subtler morphological
  signal than any hand-designed feature list, at the cost of being far less directly interpretable — "the
  17th embedding dimension" has no obvious biological meaning on its own.

High-content screening (Cell-Painting-style assays) typically multiplexes several fluorescent stains per
well — nucleus, cytoplasm, several organelles — specifically to make the handcrafted-feature route unusually
rich: far more organelle-level morphological detail per cell than a typical single-channel assay provides.
This module works with a single simulated channel to keep the code short; the profiling logic generalises
directly to many channels, simply by concatenating each channel's features into one longer profile vector.

### 📄 Research connection — Cell Painting and profile-based screening

Bray et al. (2016, *Nature Protocols*) formalised the **Cell Painting** assay: six fluorescent dyes imaged in
five channels, labeling eight broad cellular compartments at once, specifically designed to maximise the
morphological information captured per well for profiling, rather than to answer one narrow biological
question. Caicedo et al. (2017, *Nature Methods*) surveyed the data-analysis strategies this generates —
feature extraction, aggregation, normalisation — much of which this module's pipeline (Sections 2–3) mirrors
in miniature. The **JUMP-Cell Painting Consortium** (Chandrasekaran et al. 2023) scaled the idea to a public,
multi-institution dataset of over 100,000 genetic and chemical perturbations, explicitly built to study — and
correct for — exactly the batch-effect problem this module demonstrates, at an enormous scale.

## 2 · Simulating treatments, batches, and a profiling pipeline

In [ ]:
# ── 2.1 Three synthetic 'treatments' with distinct true morphological effects ──
# TREATMENT_A: a neutral/vehicle control (the baseline shape-class mixture).
# TREATMENT_B: shifts the population toward larger, more elongated cells (a true biological effect).
# TREATMENT_C: shifts the population toward irregular, more textured cells (a different true effect).
TREATMENTS = {
    "control":     dict(shape_classes=("round", "kidney"), cell_scale=1.0),
    "treatment_B": dict(shape_classes=("elongated", "dividing"), cell_scale=1.25),
    "treatment_C": dict(shape_classes=("irregular", "ring"), cell_scale=1.0),
}

# Three simulated 'plates' (batches). BATCH_SHIFT is a purely technical, non-biological difference
# (e.g. a different day's illumination/focus) applied identically to every well on that plate,
# REGARDLESS of which treatment is in the well.
BATCH_EFFECTS = {
    "plate_1": dict(background=0.05, noise_sigma=0.02),
    "plate_2": dict(background=0.14, noise_sigma=0.05),
    "plate_3": dict(background=0.22, noise_sigma=0.08),
}

N_WELLS_PER_COMBO = 6   # wells of each (treatment, batch) combination
CELLS_PER_WELL = 14
print(f"{len(TREATMENTS)} treatments x {len(BATCH_EFFECTS)} batches x {N_WELLS_PER_COMBO} wells "
      f"= {len(TREATMENTS) * len(BATCH_EFFECTS) * N_WELLS_PER_COMBO} simulated wells")


In [ ]:
# ── 2.2 Simulate every well: one image per well, with a KNOWN treatment and batch label ──
records = []
seed_counter = 10_000
for treatment, tconf in TREATMENTS.items():
    for batch, bconf in BATCH_EFFECTS.items():
        for well in range(N_WELLS_PER_COMBO):
            sim = simulate_nuclei_image(
                n_cells=CELLS_PER_WELL, size=96,
                shape_classes=tconf["shape_classes"],
                psf_sigma=1.1 * tconf["cell_scale"],
                background=bconf["background"], noise_sigma=bconf["noise_sigma"],
                seed=seed_counter)
            seed_counter += 1
            records.append(dict(treatment=treatment, batch=batch, well=well, sim=sim))

print(f"simulated {len(records)} wells")
example = [r["sim"]["image"] for r in records if r["treatment"] == "treatment_B"][:2] + \
          [r["sim"]["image"] for r in records if r["treatment"] == "control"][:2]
show_image_grid(example, titles=["treatment_B", "treatment_B", "control", "control"], ncols=4)
plt.show()


🔍 **What just happened?** Every well now has a *known* ground-truth treatment label and a *known* ground-truth batch label — the exact information a real screen's plate map would give you, but with the added advantage that we also know, by construction, exactly how large the true treatment effect and the true batch effect are.

In [ ]:
# ── 2.3 Per-cell feature extraction and per-well aggregation (the 'profiling' step) ──
def well_profile(sim):
    """Extract shape + texture features for every cell in a well (using its instance mask), and
    aggregate them into one profile vector per well (the mean over cells — the simplest aggregation;
    real pipelines also keep per-cell distributions, medians, etc.)."""
    image, mask = sim["image"], sim["instance_mask"]
    feats = []
    for cell_id in np.unique(mask):
        if cell_id == 0:
            continue
        cell_mask = mask == cell_id
        ys, xs = np.where(cell_mask)
        pad = 4
        y0, y1 = max(0, ys.min() - pad), min(image.shape[0], ys.max() + pad)
        x0, x1 = max(0, xs.min() - pad), min(image.shape[1], xs.max() + pad)
        patch = image[y0:y1, x0:x1]
        sf = shape_features(cell_mask[y0:y1, x0:x1])
        tf = glcm_features(patch)
        feats.append([sf["area"], sf["compactness"], sf["aspect_ratio"],
                      tf["contrast"], tf["energy"], tf["homogeneity"], tf["correlation"]])
    if not feats:
        return np.full(7, np.nan)
    return np.mean(feats, axis=0)

FEATURE_NAMES = ["area", "compactness", "aspect_ratio", "GLCM contrast", "GLCM energy",
                 "GLCM homogeneity", "GLCM correlation"]

for r in records:
    r["profile"] = well_profile(r["sim"])

profiles = np.array([r["profile"] for r in records])
treatments_arr = np.array([r["treatment"] for r in records])
batches_arr = np.array([r["batch"] for r in records])
print(f"profile matrix: {profiles.shape[0]} wells x {profiles.shape[1]} features")


✏️ **Your turn.** Add one more feature to `well_profile` (for instance, the
cell count in the well, `len(np.unique(mask)) - 1`, or the standard deviation of `area` across cells rather
than just its mean). Re-run the extraction. Does adding population-level variability (not just the mean)
seem like it would help or hurt the ability to tell treatments apart? Why might real profiling pipelines keep
several summary statistics per feature, not just the mean?

## 3 · Batch effects confound treatment structure

In [ ]:
# ── 3.1 PCA of the RAW profiles, coloured by treatment and by batch ──
scores, components, explained = pca(profiles, n_components=2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, labels, title in [(axes[0], treatments_arr, "coloured by TRUE treatment"),
                           (axes[1], batches_arr, "coloured by TECHNICAL batch")]:
    for lab in np.unique(labels):
        sel = labels == lab
        ax.scatter(scores[sel, 0], scores[sel, 1], label=lab, s=18, alpha=0.8)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(title); ax.legend(fontsize=8)
fig.suptitle("Raw profiles: batch structure competes with treatment structure")
fig.tight_layout(); plt.show()
print(f"explained variance ratio, PC1 & PC2: {explained}")


🔍 **What just happened?** Look carefully at both panels: if the batch effect
is strong enough, wells separate mainly by *which plate they were imaged on*, not by *which treatment they
received* — the right panel's batch clusters may look cleaner than the left panel's treatment clusters, even
though batch is pure imaging technicality and treatment is the real biology you actually care about. This is
not a hypothetical: it is one of the most common, and most consequential, failure modes in real screening
data.

In [ ]:
# ── 3.2 Quantify the confound: within-group vs. between-group distance, for each grouping ──
def group_separation(X, labels):
    """A simple separation score: mean between-group centroid distance divided by mean within-group
    spread. Higher means the grouping explains more of the variance in X (purely descriptive — not a
    formal statistical test, but enough to compare 'before' and 'after' correction)."""
    groups = {lab: X[labels == lab] for lab in np.unique(labels)}
    centroids = {lab: g.mean(axis=0) for lab, g in groups.items()}
    within = np.mean([np.linalg.norm(g - centroids[lab], axis=1).mean() for lab, g in groups.items()])
    cs = list(centroids.values())
    between = np.mean([np.linalg.norm(cs[i] - cs[j]) for i in range(len(cs)) for j in range(len(cs)) if i != j])
    return between / (within + 1e-9)

sep_treatment_raw = group_separation(profiles, treatments_arr)
sep_batch_raw = group_separation(profiles, batches_arr)
print(f"separation score by TREATMENT (raw profiles): {sep_treatment_raw:.3f}")
print(f"separation score by BATCH     (raw profiles): {sep_batch_raw:.3f}")


🧪 **Ground-truth check.** Because we injected the batch effect ourselves
(Section 2.1's `BATCH_EFFECTS`) and we know it has nothing to do with biology, any separation score at all
attributable to batch is, by construction, 100% artefact. If the batch separation score rivals or exceeds the
treatment separation score, a clustering or hit-calling analysis run on these raw profiles would be at real
risk of mistaking plate identity for biological effect — or of a batch effect accidentally cancelling out a
real but smaller treatment effect in the opposite direction.

## 4 · Correcting for batch effects

The correction this module demonstrates is deliberately simple, so that its
effect is easy to see and verify: **per-batch standardisation** (sometimes called per-plate z-scoring).
For every feature $j$ and every batch $b$, recentre and rescale using that batch's own mean and standard
deviation:

$$x'_{i,j} = \frac{x_{i,j} - \mu_{j,b(i)}}{\sigma_{j,b(i)} + \epsilon}$$

*In words:* instead of comparing raw feature values across plates, compare each well's position *relative to
its own plate's typical well* — which removes any systematic, plate-wide shift or scale change while
leaving within-plate differences (where the treatment and batch are not confounded) intact. This is a
simplification of more sophisticated batch-correction methods used in practice (Section 6), but it isolates
the essential idea clearly.

In [ ]:
# ── 4.1 Per-batch standardisation ──
corrected = np.zeros_like(profiles)
for b in np.unique(batches_arr):
    sel = batches_arr == b
    mu = profiles[sel].mean(axis=0)
    sigma = profiles[sel].std(axis=0) + 1e-9
    corrected[sel] = (profiles[sel] - mu) / sigma

scores_c, _, explained_c = pca(corrected, n_components=2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, labels, title in [(axes[0], treatments_arr, "coloured by TRUE treatment"),
                           (axes[1], batches_arr, "coloured by TECHNICAL batch")]:
    for lab in np.unique(labels):
        sel = labels == lab
        ax.scatter(scores_c[sel, 0], scores_c[sel, 1], label=lab, s=18, alpha=0.8)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(title); ax.legend(fontsize=8)
fig.suptitle("Batch-corrected profiles: treatment structure should now dominate")
fig.tight_layout(); plt.show()

sep_treatment_corr = group_separation(corrected, treatments_arr)
sep_batch_corr = group_separation(corrected, batches_arr)
print(f"separation score by TREATMENT: raw {sep_treatment_raw:.3f} -> corrected {sep_treatment_corr:.3f}")
print(f"separation score by BATCH    : raw {sep_batch_raw:.3f} -> corrected {sep_batch_corr:.3f}")


🔍 **What just happened?** After per-batch standardisation, the batch
separation score should drop substantially (the technical artefact we deliberately introduced is largely
removed), while the treatment separation score should hold up or improve relative to batch — recovering the
true biological structure that the batch effect had been competing with. Because we know both ground-truth
labels, we can *prove* this correction helped, rather than just hoping it did.

✏️ **Your turn.** Make the simulated batch effect much stronger (widen the range of `background`/
`noise_sigma` in `BATCH_EFFECTS`) or much weaker, and re-run Sections 3–4. At what relative strength does
per-batch standardisation stop being enough to recover the treatment structure? What would that tell you
about a real screen where the batch effect might be even larger than the biological effect you are looking
for?

## 5 · Hit-calling, conceptually

Once profiles are extracted and batch-corrected, **hit-calling** asks: *which
treatments produced a phenotypic effect large enough to be confident it is not just well-to-well noise?* A
simple conceptual version: compare each treatment's profile distance from the control profile against the
spread of *control-to-control* distances (replicate wells of the control treatment, which should differ only
by noise). A treatment is a plausible "hit" if its distance from control clearly exceeds the typical control-
to-control spread — the same logic as a z-score or a permutation test, applied to a multi-dimensional profile
rather than a single number.

In [ ]:
# ── 5.1 A simple hit-calling score: treatment-to-control distance vs. control-to-control spread ──
control_profiles = corrected[treatments_arr == "control"]
control_centroid = control_profiles.mean(axis=0)
control_spread = np.linalg.norm(control_profiles - control_centroid, axis=1).std()

print(f"{'treatment':14s} | mean distance from control | hit score (in control-spread units)")
for t in np.unique(treatments_arr):
    if t == "control":
        continue
    dist = np.linalg.norm(corrected[treatments_arr == t] - control_centroid, axis=1).mean()
    hit_score = dist / (control_spread + 1e-9)
    flag = "  <-- HIT" if hit_score > 3 else ""
    print(f"{t:14s} | {dist:26.3f} | {hit_score:6.2f}{flag}")


✏️ **Your turn.** Add a fourth, *null* treatment that uses the exact same
simulation parameters as `control` (no true biological effect at all) and re-run hit-calling. Does it
correctly score as a non-hit? This is the simplest possible **negative control** check for a hit-calling
pipeline, and real screens always run several.

## 6 · From toy to real data

To move from this notebook's simplified pipeline to a real profiling workflow:

- **Bray, M.-A. et al. (2016). Cell Painting, a high-content image-based assay for morphological profiling
  using multiplexed fluorescent dyes.** *Nature Protocols*, 11, 1757–1774. The standard assay this module's
  simulated "treatments" are a miniature stand-in for.
- **Caicedo, J. C. et al. (2017). Data-analysis strategies for image-based cell profiling.** *Nature
  Methods*, 14, 849–863. Surveys feature extraction, aggregation and normalisation strategies, including more
  sophisticated batch-correction approaches (e.g. quantile normalisation, Harmony/ComBat-style methods)
  beyond the per-batch standardisation this module demonstrated.
- **Chandrasekaran, S. N. et al. (2023). JUMP Cell Painting dataset.** A public, multi-institution dataset of
  over 100,000 chemical and genetic perturbations, purpose-built to study batch effects and enable profiling
  methods development at scale.
- **CellProfiler** (Carpenter et al. 2006; cellprofiler.org) is the production, open-source tool most
  commonly used to extract exactly these kinds of per-cell features from multiplexed microscopy images at
  screening scale.

## 📌 Key takeaways

- Image-based profiling turns each cell or well into a feature vector (a profile), not a single label — the
  natural unit of comparison for high-content screening and perturbation biology.
- Classical handcrafted features and learned deep embeddings are two ways to build a profile, trading
  interpretability against representational richness.
- A batch effect is a systematic, non-biological difference between plates/sessions that can dominate a raw
  profile's low-dimensional structure, confounding or masking real treatment effects.
- Per-batch standardisation is a simple, verifiable correction; its effectiveness should always be checked
  against known ground truth or negative controls, not assumed.
- Hit-calling compares a treatment's profile shift against the natural spread of replicate control wells —
  always validate it with a true negative control.

## ✅ Checkpoint

1. Why is a feature vector (a profile) a more natural output for a screening assay than a single
   classification label?
   <details><summary>Answer</summary>A profile captures the full morphological state of the population,
   supporting downstream comparisons the screen's designer did not anticipate in advance (clustering by
   mechanism, computing distance to known reference compounds), whereas a single label only answers one
   predefined question.</details>

2. In Section 3, why can a batch effect "compete with" or even dominate a real treatment effect in a PCA
   plot of the raw profiles?
   <details><summary>Answer</summary>If the batch-to-batch shift in feature values is larger than the
   treatment-to-treatment shift, the directions of greatest variance (what PCA captures) will align more with
   batch than with treatment, making batch the dominant visible structure even though it carries no biological
   information.</details>

3. What ground truth did this notebook have access to that a real screening lab typically does NOT, and why
   does that matter for interpreting the "before vs. after correction" result?
   <details><summary>Answer</summary>The notebook knows the exact true treatment effect and the exact true
   batch effect because both were simulated. A real lab must validate batch correction using proxies —
   negative controls, replicate consistency, known reference compounds — rather than ever seeing the
   "ground truth" directly.</details>

4. Why does hit-calling compare a treatment's effect size against the spread of *control* replicates, rather
   than against a fixed numerical threshold?
   <details><summary>Answer</summary>The natural well-to-well noise level differs across assays, feature
   sets and screens; a fixed threshold would be too strict for noisy assays and too lenient for clean ones,
   whereas comparing against the control replicate spread automatically calibrates the threshold to the
   assay's own noise level.</details>

## 🧠 Questions & discussion

1. Per-batch standardisation assumes every batch has a similar mix of treatments (so that the batch's mean
   is dominated by "typical" wells, not skewed by one unusually strong treatment). When might that assumption
   fail in a real screening plate layout, and what would go wrong if it did?
2. Learned deep embeddings (Modules 08–09) can capture subtler morphological signal than handcrafted
   features, but are harder to audit for batch effects (you cannot easily ask "which embedding dimension is
   the batch artefact?"). How would you check a deep-embedding profile for a hidden batch confound?
3. If a compound's profile is a near-perfect match to a reference compound with a known mechanism, what would
   you want to verify before concluding the two compounds share that mechanism?

## 🚀 Bonus — going further

1. Implement a second batch-correction method — e.g. subtracting only the per-batch mean (no rescaling by
   the standard deviation) — and compare its effect on the separation scores to the full standardisation used
   above.
2. Simulate a *plate-position* effect (e.g. wells near the plate edge are systematically different from
   centre wells, a classic real artefact) in addition to the per-plate batch effect, and see whether per-batch
   standardisation alone is enough to correct for it, or whether you need a position-aware correction too.
3. Build a simple nearest-centroid classifier that predicts a well's treatment from its batch-corrected
   profile, held out from the wells used to compute each treatment's centroid, and report its accuracy as a
   more rigorous substitute for eyeballing the PCA plot.

## 📝 Lab note

```text
Module 22 — Image-based profiling and perturbation biology
Date:
Separation score by treatment, raw vs. corrected:
Separation score by batch, raw vs. corrected:
Hit-calling scores per treatment:
Any treatment whose effect was masked or exaggerated by the batch effect, before correction:
```

## 📚 References

1. Bray, M.-A. et al. (2016). Cell Painting, a high-content image-based assay for morphological profiling
   using multiplexed fluorescent dyes. *Nature Protocols*, 11, 1757–1774.
2. Caicedo, J. C. et al. (2017). Data-analysis strategies for image-based cell profiling. *Nature Methods*,
   14, 849–863.
3. Chandrasekaran, S. N. et al. (2023). JUMP Cell Painting dataset: morphological impact of 136,000
   chemical and genetic perturbations. *bioRxiv / Nature Methods*.
4. Carpenter, A. E. et al. (2006). CellProfiler: image analysis software for identifying and quantifying
   cell phenotypes. *Genome Biology*, 7, R100.
5. Johnson, W. E., Li, C. & Rabinovic, A. (2007). Adjusting batch effects in microarray expression data
   using empirical Bayes methods (ComBat) — the batch-correction method this module's standardisation
   simplifies.
6. Ljosa, V., Sokolnicki, K. L. & Carpenter, A. E. (2012). Annotated high-throughput microscopy image sets
   for validation. *Nature Methods*, 9, 637.

## ➡️ Next

Module 23 scales the same "profile many objects, then look for structure" logic from individual cells in a
well to many cell types arranged in space within a tissue section — spatial omics and multiplexed imaging —
directly extending Module 18's spatial statistics to a many-channel, many-cell-type setting.